# 分類・生成タスクの評価尺度と性質

書籍 第1章「自然言語処理システムのデザイン」（1-4 評価：評価尺度とエラー分析、練習問題 2 など）に対応する演習ノートブックです。
CPU のみ・数秒で実行でき、以下の内容を定義式からの手計算（スクラッチ実装）と定番ライブラリの比較を通じて体験します。

1. 分類タスクの評価尺度（正解率・適合率・再現率・F値）の計算と `scikit-learn` との一致確認
2. 不均衡データにおける「正解率の罠」と Macro-F1 / Micro-F1 の違い
3. 生成タスクの評価尺度（BLEU）の仕組み（修正 n-gram 適合率・クリッピング・簡潔性ペナルティ）と `sacrebleu` との一致確認
4. 表層一致に基づく評価尺度（BLEU など）の限界（言い換え・語順・意味の違い）

## 1. 準備

生成タスクの評価尺度 BLEU を計算するために `sacrebleu` をインストールします（`numpy`, `pandas`, `scikit-learn` は Google Colab に標準で入っています）。

In [1]:
!pip install -q sacrebleu

In [2]:
import math
from collections import Counter

import numpy as np
import pandas as pd
import sacrebleu
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

---

## 2. 分類タスクの評価尺度（定義式の実装と検証）

2値分類（正例 `1` / 負例 `0`）の予測結果から、混同行列の 4 つの要素（TP, FP, FN, TN）を数え、定義式どおりに各指標を計算します。

- **正解率（Accuracy）**: $\displaystyle \frac{TP + TN}{TP + FP + FN + TN}$
- **適合率（Precision）**: $\displaystyle \frac{TP}{TP + FP}$（正例と予測したもののうち実際に正例だった割合）
- **再現率（Recall）**: $\displaystyle \frac{TP}{TP + FN}$（実際の正例のうち正しく正例と予測できた割合）
- **F値（F1 score）**: $\displaystyle \frac{2 \cdot \text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$（適合率と再現率の調和平均）

In [3]:
def compute_binary_metrics(y_true, y_pred):
    """混同行列から Accuracy / Precision / Recall / F1 を計算する"""
    tp = sum(t == 1 and p == 1 for t, p in zip(y_true, y_pred))
    fp = sum(t == 0 and p == 1 for t, p in zip(y_true, y_pred))
    fn = sum(t == 1 and p == 0 for t, p in zip(y_true, y_pred))
    tn = sum(t == 0 and p == 0 for t, p in zip(y_true, y_pred))

    acc = (tp + tn) / (tp + fp + fn + tn)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return {"TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "accuracy": acc, "precision": prec, "recall": rec, "f1": f1}

y_true = [1, 1, 1, 1, 1, 0, 0, 0, 0, 0]
y_pred = [1, 1, 1, 0, 0, 1, 0, 0, 0, 0]

m = compute_binary_metrics(y_true, y_pred)
print(f"TP={m['TP']}, FP={m['FP']}, FN={m['FN']}, TN={m['TN']}")
print(f"手計算: Accuracy={m['accuracy']:.4f}, Precision={m['precision']:.4f}, Recall={m['recall']:.4f}, F1={m['f1']:.4f}")
print(
    f"sklearn: Accuracy={accuracy_score(y_true, y_pred):.4f}, "
    f"Precision={precision_score(y_true, y_pred):.4f}, "
    f"Recall={recall_score(y_true, y_pred):.4f}, "
    f"F1={f1_score(y_true, y_pred):.4f}"
)

TP=3, FP=1, FN=2, TN=4
手計算: Accuracy=0.7000, Precision=0.7500, Recall=0.6000, F1=0.6667
sklearn: Accuracy=0.7000, Precision=0.7500, Recall=0.6000, F1=0.6667


---

## 3. 不均衡データにおける「正解率の罠」

実際の NLP タスク（スパム検出、有害表現検出、希少な誤りの検出など）では、クラスの比率が大きく偏っている（**不均衡データ**）ことがよくあります。

100件のうち **95件が負例（0）、5件だけが正例（1）** というデータで、以下の 2 つのモデルを比較してみましょう。
- **モデル A（ダミー分類器）**: 入力を見ずに、すべて `0`（負例）と予測する
- **モデル B（実用的な分類器）**: 正例 5件のうち 4件を検出し、誤検出（FP）が 3件だけある

In [4]:
y_imb = [1] * 5 + [0] * 95
pred_dummy = [0] * 100
pred_model = [1, 1, 1, 1, 0] + [1, 1, 1] + [0] * 92

rows = []
for name, pred in [("モデルA（すべて0と予測）", pred_dummy), ("モデルB（正例を検出）", pred_model)]:
    rows.append({
        "モデル": name,
        "Accuracy": round(accuracy_score(y_imb, pred), 3),
        "Precision (正例)": round(precision_score(y_imb, pred, zero_division=0), 3),
        "Recall (正例)": round(recall_score(y_imb, pred, zero_division=0), 3),
        "F1 (正例)": round(f1_score(y_imb, pred, zero_division=0), 3),
        "Macro-F1": round(f1_score(y_imb, pred, average="macro", zero_division=0), 3),
    })

pd.DataFrame(rows)

,モデル,Accuracy,Precision (正例),Recall (正例),F1 (正例),Macro-F1
0,モデルA（すべて0と予測）,0.95,0.000,0.0,0.000,0.487
1,モデルB（正例を検出）,0.96,0.571,0.8,0.667,0.823


モデル A は何も検出していないにもかかわらず **Accuracy が 0.950** と非常に高く見えますが、正例の F1 は **0.000**、Macro-F1 も **0.487** にとどまります。不均衡データでは Accuracy だけでなく、適合率・再現率・F1（または Macro-F1）をあわせて確認することが重要です。

---

## 4. 多クラス分類の Macro 平均と Micro 平均

3クラス分類（`pos` / `neu` / `neg`）では、クラスごとの F1 をどのように集約するかによって値が大きく変わります。

- **Macro 平均**: 各クラスの F1 を**クラス数で単純平均**する（データ数が少ないクラスも対等に扱う）
- **Micro 平均**: 全クラスの TP・FP・FN を**合算してから**計算する（各事例を対等に扱うため、単一ラベル多クラス分類では Accuracy と一致する）

In [5]:
labels = ["pos", "neu", "neg"]
y_mc_true = ["pos"] * 40 + ["neu"] * 10 + ["neg"] * 10
y_mc_pred = (
    ["pos"] * 38 + ["neu"] * 2 +
    ["neu"] * 3  + ["pos"] * 7 +
    ["neg"] * 4  + ["pos"] * 6
)

print("混同行列（行: 正解 / 列: 予測）:")
display(pd.DataFrame(confusion_matrix(y_mc_true, y_mc_pred, labels=labels), index=labels, columns=labels))
print(classification_report(y_mc_true, y_mc_pred, labels=labels, digits=4))
print(f"Accuracy = {accuracy_score(y_mc_true, y_mc_pred):.4f}")
print(f"Micro-F1 = {f1_score(y_mc_true, y_mc_pred, average='micro'):.4f}")
print(f"Macro-F1 = {f1_score(y_mc_true, y_mc_pred, average='macro'):.4f}")

混同行列（行: 正解 / 列: 予測）:


,pos,neu,neg
pos,38,2,0
neu,7,3,0
neg,6,0,4


              precision    recall  f1-score   support

         pos     0.7451    0.9500    0.8352        40
         neu     0.6000    0.3000    0.4000        10
         neg     1.0000    0.4000    0.5714        10

    accuracy                         0.7500        60
   macro avg     0.7817    0.5500    0.6022        60
weighted avg     0.7634    0.7500    0.7187        60

Accuracy = 0.7500
Micro-F1 = 0.7500
Macro-F1 = 0.6022


---

## 5. 生成タスクの評価尺度：BLEU の仕組み

機械翻訳や要約などの生成タスクでは、正解が 1 つに定まらないため、参照文（リファレンス）との n-gram の重なり具合を測る **BLEU** が広く使われてきました。

BLEU（1〜4-gram）は次の 2 つの要素の積で定義されます。

1. **修正 n-gram 適合率（Modified n-gram precision） $p_n$**:
   同じ単語を繰り返して適合率を水増しできないよう、各 n-gram の一致回数を「参照文での最大出現回数」で上限（クリッピング）します。
2. **簡潔性ペナルティ（Brevity Penalty; BP）**:
   短い文だけを出力して適合率を稼ぐのを防ぐため、出力長 $c$ が参照文長 $r$ より短い場合に $\exp(1 - r/c)$ のペナルティを掛けます。

$$\text{BLEU} = \text{BP} \cdot \exp\left(\frac{1}{4}\sum_{n=1}^{4} \log p_n\right), \quad \text{BP} = \begin{cases} 1 & (c > r) \\ \exp(1 - r/c) & (c \le r) \end{cases}$$

In [6]:
def ngrams(tokens, n):
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]

def compute_bleu(hypotheses, references, max_n=4):
    """コーパスレベルの BLEU (smooth_method='none' 相当) を定義式から計算する"""
    clipped_counts = [0] * max_n
    total_counts = [0] * max_n
    hyp_len = 0
    ref_len = 0

    for hyp, ref in zip(hypotheses, references):
        h_tok = hyp.split()
        r_tok = ref.split()
        hyp_len += len(h_tok)
        ref_len += len(r_tok)

        for n in range(1, max_n + 1):
            h_ng = Counter(ngrams(h_tok, n))
            r_ng = Counter(ngrams(r_tok, n))
            clipped_counts[n - 1] += sum(min(cnt, r_ng[ng]) for ng, cnt in h_ng.items())
            total_counts[n - 1] += sum(h_ng.values())

    precisions = [c / t if t > 0 else 0.0 for c, t in zip(clipped_counts, total_counts)]
    bp = 1.0 if hyp_len > ref_len else math.exp(1.0 - ref_len / hyp_len)
    if min(precisions) == 0.0:
        bleu = 0.0
    else:
        bleu = bp * math.exp(sum(math.log(p) for p in precisions) / max_n)
    return bleu * 100.0, [p * 100.0 for p in precisions], bp

refs = [
    "the cat is on the mat",
    "there is a cat on the mat",
]
hyps = [
    "the cat is on the mat",
    "there is a dog on the mat",
]

my_bleu, precs, bp = compute_bleu(hyps, refs)
lib_bleu = sacrebleu.corpus_bleu(hyps, [refs], tokenize="none", smooth_method="none")
print(f"手計算 BLEU   = {my_bleu:.4f} (n-gram 適合率={[round(p, 2) for p in precs]}, BP={bp:.4f})")
print(f"sacrebleu     = {lib_bleu.score:.4f}")

手計算 BLEU   = 68.1561 (n-gram 適合率=[92.31, 81.82, 66.67, 42.86], BP=1.0000)
sacrebleu     = 68.1561


### クリッピングと簡潔性ペナルティの働きを確認する

もしクリッピングや簡潔性ペナルティがなかったらどうなるかを、極端な出力例で確認してみましょう。

In [7]:
ref_sample = ["the cat is on the mat"]
cases = [
    ("適切な翻訳（完全一致）", ["the cat is on the mat"]),
    ("同じ語を連呼（クリッピングの検証）", ["the the the the the the"]),
    ("極端に短い出力（BP の検証）", ["the cat"]),
]

for label, h in cases:
    b_none = sacrebleu.corpus_bleu(h, [ref_sample], tokenize="none", smooth_method="none")
    b_exp = sacrebleu.corpus_bleu(h, [ref_sample], tokenize="none", smooth_method="exp")
    print(f"{label:32s} | BLEU(none)={b_none.score:6.2f}, BLEU(smooth)={b_exp.score:6.2f}, BP={b_none.bp:.3f}, p1={b_none.precisions[0]:.1f}")

適切な翻訳（完全一致）                      | BLEU(none)=100.00, BLEU(smooth)=100.00, BP=1.000, p1=100.0
同じ語を連呼（クリッピングの検証）                | BLEU(none)=  0.00, BLEU(smooth)=  9.65, BP=1.000, p1=33.3
極端に短い出力（BP の検証）                  | BLEU(none)=  0.00, BLEU(smooth)=  0.00, BP=0.135, p1=100.0


- `"the the the the the the"` は 6 単語すべてが参照文の `"the"` と一致するため、クリッピングなしでは 1-gram 適合率が 100% になってしまいますが、クリッピングにより参照文中の出現回数（2回）までに制限され、`p1 = 33.3`、2-gram 以降は 0 のため BLEU は低くなります。
- `"the cat"` は 1-gram・2-gram 適合率が 100% ですが、文長が短すぎるため `BP = 0.135` と強いペナルティが掛かります。

---

## 6. 表層一致に基づく評価尺度の限界

BLEU は高速で再現性が高い一方、**表層的な n-gram の一致**しか見ないため、意味の近さや文法的自然さを直接測ることはできません。いくつかの典型例でスコアを比べてみましょう。

In [8]:
ref_en = ["I bought a new book at the bookstore yesterday"]
examples = [
    ("参照文と同一", "I bought a new book at the bookstore yesterday"),
    ("意味は同じ（言い換え）", "Yesterday I purchased a brand new book from the shop"),
    ("語順を崩した非流暢な文", "yesterday book new a bought I bookstore the at"),
    ("否定が逆（意味は真逆）", "I did not buy a new book at the bookstore yesterday"),
]

eval_rows = []
for desc, hyp in examples:
    res = sacrebleu.corpus_bleu([hyp], [ref_en], tokenize="none")
    eval_rows.append({
        "ケース": desc,
        "出力文": hyp,
        "BLEU": round(res.score, 2),
        "1-gram": round(res.precisions[0], 1),
        "4-gram": round(res.precisions[3], 1),
    })

pd.DataFrame(eval_rows)

,ケース,出力文,BLEU,1-gram,4-gram
0,参照文と同一,I bought a new book at the bookstore yesterday,100.00,100.0,100.0
1,意味は同じ（言い換え）,Yesterday I purchased a brand new book from th...,10.55,50.0,3.6
2,語順を崩した非流暢な文,yesterday book new a bought I bookstore the at,8.26,100.0,2.1
3,否定が逆（意味は真逆）,I did not buy a new book at the bookstore yest...,59.00,72.7,50.0


上の表から、以下の重要な性質が読み取れます。

- **言い換えに厳しい**: 意味がほぼ同じでも異なる単語（`bought` → `purchased`、`bookstore` → `shop`）を使うと BLEU が大きく下がります。
- **否定や重要語の反転に鈍感**: `"not"` を1語足して意味が真逆になっても、多くの n-gram が一致するため高い BLEU がついてしまいます。

このような表層一致の限界を補うために、事前学習言語モデルの埋め込み類似度を使う評価尺度や、参照文を使わずに品質を推定する手法（第2章の[品質推定の演習](https://colab.research.google.com/github/mamorlis/nlpbook/blob/main/notebooks/ch2_qe.ipynb)）が研究・活用されています。